<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_02_ridge_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_02 - SEQ2ONE - Modelo Ridge**

**Introducción**

Luego de establecer el baseline Naive, se introduce **Ridge Regression** como
primer modelo **entrenable** del stage_07 bajo un enfoque **seq2one**.

Ridge es una regresión lineal con **regularización L2**, diseñada para manejar
vectores de alta dimensión y features correlacionadas, manteniendo un control
explícito de la complejidad del modelo.

En este proyecto, Ridge permite evaluar cuánto valor puede capturarse mediante
una relación lineal directa entre la ventana histórica intradía
(L x N → cantidad de features) y el target escalar futuro.

---

**Rol en el pipeline**

- Primer modelo con capacidad de aprendizaje real.
- Referencia lineal fuerte y estable.
- Punto de comparación obligatorio para modelos no lineales posteriores.
- Si modelos más complejos no superan a Ridge en VALID, su aporte es cuestionable.

---

**Esquema general**

- **Entrada (X):** vector aplanado de LxN features.
- **Salida (Y):** valor escalar futuro.
- **Entrenamiento:** conjunto TRAIN.
- **Evaluación:** conjunto VALID.
- **Regularización:** L2 (controlada por el parámetro \(\alpha\)).

Las métricas obtenidas se almacenan como artefactos y se utilizan posteriormente en el **stage_07** para la comparación final entre modelos.


## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [3]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [4]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [5]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [7]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [8]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de ventanas**

In [9]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    with np.load(path, allow_pickle=False) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Cast explícito (si desea forzarlo)
    X = np.asarray(X, dtype=np.float16)
    y = np.asarray(y, dtype=np.float32)

    return X, y

In [10]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [11]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [12]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [13]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,  # <-- NUEVO
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

In [14]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths, flatten_X = False)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**

In [19]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [20]:
def get_metrics (bundle, model):

    # -------- VALID --------
    X_valid = bundle ['valid']['X']
    y_pred_valid = model.predict(X_valid)

    # --------TEST --------
    X_test = bundle ['test']['X']
    y_pred_test = model.predict(X_test)

    y_valid = bundle["valid"]["y"]
    y_test  = bundle["test"]["y"]

    metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)
    metrics_test  = compute_seq2one_metrics(y_test,  y_pred_test,  compute_r2=True)

    return metrics_valid, metrics_test

## **9. Gestión de dataset de métricas**

In [21]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [22]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Modelo Ridge Regression (seq2one)**

**Idea básica**

**Ridge Regression** es una regresión lineal con **regularización L2**.  
Aprende un vector de pesos **w** que relaciona linealmente la entrada aplanada
con el target escalar, penalizando pesos grandes para reducir el sobreajuste.

Formalmente:

$$
\hat{y}_t = \mathbf{w}^\top \mathbf{x}_t + b
$$

con función objetivo:

$$
\min_{\mathbf{w}, b}
\sum_t (y_t - \hat{y}_t)^2
\;+\;
\alpha \sum_i w_i^2
$$

donde $\alpha\$ controla la **fuerza de la regularización**.

---

**Regularización (Ridge / Lasso)**

- **Ridge (L2):**
  - Penaliza el cuadrado de los coeficientes.
  - Reduce la magnitud de los pesos sin anularlos.
- **Lasso (L1):**
  - Penaliza el valor absoluto de los coeficientes.
  - Puede llevar pesos exactamente a cero (sparsity).

**Riesgo:** Bajo, controlado por diseño.  
La regularización es **intrínseca al modelo** y está gobernada por el
hiperparámetro \(\alpha\).

No se requieren técnicas adicionales como **dropout** o **early stopping**,
ya que no se trata de un modelo iterativo por épocas.

---

**Por qué Ridge encaja bien en este proyecto**

- Entrada de **alta dimensión**: 60 × 20 = **1200 features**.
- Features **altamente correlacionadas** (estructura temporal).
- Modelo:
  - simple,
  - estable,
  - rápido de entrenar,
  - interpretable como referencia lineal.

Ridge actúa como el **baseline entrenable** contra el cual se comparan
modelos más complejos (MLP, LSTM, TCN, Transformer).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- `alpha`: fijo (por ejemplo, `1.0`)
- `fit_intercept`: `True`
- `random_state`: no aplica
- **Sin validación interna** (la evaluación se realiza externamente en VALID)

El ajuste fino del coeficiente de regularización se aborda en etapas posteriores.


### **10.2. Implementación del modelo**

In [23]:
from sklearn.linear_model import Ridge

def build_ridge_model(*, alpha: float = 1.0) -> Ridge:
    """
    Construye un modelo Ridge Regression para seq2one.

    Parámetros
    ----------
    alpha : float
        Fuerza de regularización L2.

    Retorna
    -------
    model : sklearn.linear_model.Ridge
        Modelo Ridge configurado.
    """
    model = Ridge(
        alpha=alpha,
        fit_intercept=True,
        solver="sag",
        random_state=42,  # o "lsqr"
    )
    return model


### **10.3. Entrenamiento (TRAIN)**

In [24]:
import time
import numpy as np
from sklearn.linear_model import Ridge

def _ts() -> str:
    return time.strftime("%H:%M:%S")

def train_ridge_for_bundle(bundle: dict, *, alpha: float = 1.0, solver: str = "auto", verbose: bool = True):
    """
    Entrena Ridge sobre bundle['train'].
    Instrumentado para ver dónde se va el tiempo.
    """
    t0_all = time.perf_counter()

    # 1) Extraer data
    t0 = time.perf_counter()
    X = bundle["train"]["X"]
    y = bundle["train"]["y"]
    dt_extract = time.perf_counter() - t0

    # 2) Asegurar dtype razonable (evita float16 en fit, y evita float64 por defecto)
    #    (Ridge + sklearn suelen andar mejor con float32/float64; float16 puede disparar conversions internas)
    t0 = time.perf_counter()
    if X.dtype == np.float16:
        X = X.astype(np.float32, copy=False)
    elif X.dtype not in (np.float32, np.float64):
        X = X.astype(np.float32, copy=False)

    if y.dtype != np.float32 and y.dtype != np.float64:
        y = y.astype(np.float32, copy=False)
    dt_cast = time.perf_counter() - t0

    # 3) Fit Ridge
    t0 = time.perf_counter()
    model = Ridge(alpha=alpha, solver=solver, random_state=0)
    model.fit(X, y)
    dt_fit = time.perf_counter() - t0

    dt_all = time.perf_counter() - t0_all

    if verbose:
        print(f"[{_ts()}]   [RIDGE-FIT] extract={dt_extract:.2f}s | cast={dt_cast:.2f}s | fit={dt_fit:.2f}s | total={dt_all:.2f}s | solver={solver} | X={X.shape} {X.dtype}")

    return model

In [25]:
#ridge_60 = train_ridge_for_bundle(bundle_60, alpha=1.0)
#ridge_90 = train_ridge_for_bundle(bundle_90, alpha=1.0)

## **11. Ejecución completa**

In [26]:
import time
import gc
import pandas as pd

def _ts() -> str:
    return time.strftime("%H:%M:%S")

def run_ridge(window_size: int, *, alpha: float = 1.0, verbose: bool = True) -> pd.DataFrame:
    """
    Ridge SEQ2ONE por window_size, recorriendo 4 targets.
    Imprime tiempos por etapa con formato:
      [HH:MM:SS] [i/4] START target='delta_60' | L30
      [HH:MM:SS]   [BUILD] ... dt=...
      [HH:MM:SS]   [TRAIN] ... dt=...
      [HH:MM:SS]   [EVAL]  ... dt=...
    """
    L = int(window_size)
    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{L} | alpha={alpha}")
        print("=" * 80)

    for i, target in enumerate(targets, start=1):
        if verbose:
            print(f"[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{L}")

        # =========================================================
        # 1) BUILD TRAIN BUNDLE (suele ser el cuello si lee NPZ y descomprime)
        # =========================================================
        t0 = time.perf_counter()
        if verbose:
            print(f"[{_ts()}]   [BUILD] Creando bundle TRAIN (flatten_X=True) ...")

        (bundle_train,) = create_bundles(
            window_size=L,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,
            splits=("train",),
        )

        dt_build_train = time.perf_counter() - t0
        if verbose:
            Xt, yt = bundle_train["train"]["X"], bundle_train["train"]["y"]
            print(f"[{_ts()}]   [BUILD] OK | train X={Xt.shape} y={yt.shape} | dt={dt_build_train:.2f}s")

        # =========================================================
        # 2) TRAIN (para Ridge suele ser relativamente rápido)
        # =========================================================
        t0 = time.perf_counter()
        if verbose:
            print(f"[{_ts()}]   [TRAIN] Entrenando Ridge ...")

        model = train_ridge_for_bundle(bundle_train, alpha=alpha, solver="lsqr", verbose=verbose)

        dt_train = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [TRAIN] OK | dt={dt_train:.2f}s")

        # Metadatos (use bundle_train; valid/test comparten estos valores)
        horizon = bundle_train["horizon"]
        ws = bundle_train["window_size"]
        tgt = bundle_train["target"]

        # Liberar bundle_train (ahorra RAM)
        del bundle_train
        gc.collect()

        # =========================================================
        # 3) BUILD EVAL BUNDLE (valid+test) (también puede ser pesado)
        # =========================================================
        t0 = time.perf_counter()
        if verbose:
            print(f"[{_ts()}]   [BUILD] Creando bundle EVAL (valid/test, flatten_X=True) ...")

        (bundle_eval,) = create_bundles(
            window_size=L,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,
            splits=("valid", "test"),
        )

        dt_build_eval = time.perf_counter() - t0
        if verbose:
            Xv, yv = bundle_eval["valid"]["X"], bundle_eval["valid"]["y"]
            Xte, yte = bundle_eval["test"]["X"], bundle_eval["test"]["y"]
            print(f"[{_ts()}]   [BUILD] OK | valid X={Xv.shape} y={yv.shape} | test X={Xte.shape} y={yte.shape} | dt={dt_build_eval:.2f}s")

        # =========================================================
        # 4) EVAL METRICS (puede ser pesado si predice sobre millones)
        # =========================================================
        t0 = time.perf_counter()
        if verbose:
            print(f"[{_ts()}]   [EVAL] Calculando métricas (valid/test) ...")

        metrics_valid, metrics_test = get_metrics(bundle_eval, model)

        dt_eval = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [EVAL] OK | dt={dt_eval:.2f}s")

        # Liberar bundle_eval
        del bundle_eval
        gc.collect()

        # =========================================================
        # 5) DF (normalmente despreciable)
        # =========================================================
        t0 = time.perf_counter()

        rows.append(metrics_to_df(metrics_valid, model="ridge", split="valid", horizon=horizon, window_size=ws, target=tgt))
        rows.append(metrics_to_df(metrics_test,  model="ridge", split="test",  horizon=horizon, window_size=ws, target=tgt))

        dt_df = time.perf_counter() - t0

        # Liberar lo que queda
        del model
        del metrics_valid, metrics_test
        gc.collect()

        if verbose:
            total_target = dt_build_train + dt_train + dt_build_eval + dt_eval + dt_df
            print(
                f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | L{L} | "
                f"dt_total={total_target:.2f}s | "
                f"build_train={dt_build_train:.2f}s | train={dt_train:.2f}s | "
                f"build_eval={dt_build_eval:.2f}s | eval={dt_eval:.2f}s | df={dt_df:.2f}s"
            )

    # Consolidar DF final
    df_ridge_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        print(f"[{_ts()}] [DONE] L{L} | rows={len(df_ridge_metrics)}")

    return df_ridge_metrics

In [27]:
import time
import pandas as pd

def _ts() -> str:
    return time.strftime("%H:%M:%S")

def run_ridge_incremental(
    window_sizes: list[int],
    *,
    alpha: float = 1.0,
    name: str = "ridge",  # -> guarda como seq2one_{name}_metrics.parquet
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Igual que su versión, pero con logs con timestamp tipo:
      [01:30:03] [LOAD] ...
      [01:30:14] [RUN] ...
      [01:31:10] [DONE] ...
    """

    t_total0 = time.perf_counter()

    # 1) Cargar si existe (I/O a Drive suele ser lento)
    t0 = time.perf_counter()
    if verbose:
        print(f"[{_ts()}] [LOAD] Leyendo métricas existentes (name='{name}') ...")
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)
    dt_load = time.perf_counter() - t0
    if verbose:
        print(f"[{_ts()}] [LOAD] OK | rows={len(df_all)} | dt={dt_load:.2f}s")

    # 2) Asegurar estructura base
    if df_all.empty:
        df_all = pd.DataFrame(columns=[
            "model", "split", "window_size", "target", "horizon_min",
            "MAE", "RMSE", "R2", "DA", "alpha"
        ])
    if "alpha" not in df_all.columns:
        df_all["alpha"] = pd.NA

    key_cols = ["model", "alpha", "window_size", "target", "split", "horizon_min"]

    # 3) Normalizar tipos
    t0 = time.perf_counter()
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")
    dt_norm = time.perf_counter() - t0
    if verbose:
        print(f"[{_ts()}] [NORM] dtype window_size/horizon_min | dt={dt_norm:.2f}s")

    # 4) Loop por window_size
    for idx, ws in enumerate(window_sizes, start=1):
        t_ws0 = time.perf_counter()

        # 4.1) Chequeo skip
        t0 = time.perf_counter()
        df_ws = df_all[
            (df_all["model"] == "ridge") &
            (df_all["alpha"] == alpha) &
            (df_all["window_size"] == ws)
        ]
        dt_filter = time.perf_counter() - t0

        if len(df_ws) >= 8:
            if verbose:
                print(f"[{_ts()}] [{idx}/{len(window_sizes)}] [SKIP] L{ws} | rows={len(df_ws)} | dt_filter={dt_filter:.2f}s")
            continue

        if verbose:
            print("\n" + "=" * 90)
            print(f"[{_ts()}] [{idx}/{len(window_sizes)}] [RUN] RIDGE incremental | L{ws} | alpha={alpha}")
            print("=" * 90)

        # 5) run_ridge (principal costo)
        t0 = time.perf_counter()
        df_new = run_ridge(ws, alpha=alpha, verbose=verbose).copy()
        dt_run = time.perf_counter() - t0
        df_new["alpha"] = alpha
        if verbose:
            print(f"[{_ts()}]   [RIDGE] run_ridge(L{ws}) OK | new_rows={len(df_new)} | dt={dt_run:.2f}s")

        # 6) Anti-duplicados
        t0 = time.perf_counter()
        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()
        dt_dedupe = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [DEDUPE] kept_rows={len(df_new)} | dt={dt_dedupe:.2f}s")

        if df_new.empty:
            if verbose:
                print(f"[{_ts()}]   [INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        # 7) Merge + dedupe
        t0 = time.perf_counter()
        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)
        dt_merge = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [MERGE] total_rows={len(df_all)} | dt={dt_merge:.2f}s")

        # 8) Guardar checkpoint (I/O)
        t0 = time.perf_counter()
        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)
        dt_save = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [SAVE] checkpoint OK | dt={dt_save:.2f}s")

        dt_ws = time.perf_counter() - t_ws0
        if verbose:
            print(
                f"[{_ts()}] [{idx}/{len(window_sizes)}] [DONE] L{ws} | "
                f"dt_total={dt_ws:.2f}s | filter={dt_filter:.2f}s | run={dt_run:.2f}s | "
                f"dedupe={dt_dedupe:.2f}s | merge={dt_merge:.2f}s | save={dt_save:.2f}s"
            )

    dt_total = time.perf_counter() - t_total0
    if verbose:
        print(f"\n[{_ts()}] [TOTAL] run_ridge_incremental | dt={dt_total:.2f}s")

    return df_all

In [28]:
#df_ridge_all_sizes = load_seq2one_metrics_if_exists(name="ridge", base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")

In [29]:
#RIDGE_ALL_TRAIN = '''
df_ridge_all_sizes = run_ridge_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    alpha=1.0,
    name="ridge",   # genera seq2one_ridge_metrics.parquet
    verbose=True,
)

df_ridge_all_sizes
#'''

[02:05:59] [LOAD] Leyendo métricas existentes (name='ridge') ...
[02:05:59] [LOAD] OK | rows=0 | dt=0.00s
[02:05:59] [NORM] dtype window_size/horizon_min | dt=0.00s

[02:05:59] [1/5] [RUN] RIDGE incremental | L30 | alpha=1.0

RIDGE | SEQ2ONE | WINDOW_SIZE=L30 | alpha=1.0
[02:05:59] [1/4] START target='delta_60' | L30
[02:05:59]   [BUILD] Creando bundle TRAIN (flatten_X=True) ...
H60 Train: (463872, 1080) (463872,)
Scaler H60: StandardScaler
[02:06:04]   [BUILD] OK | train X=(463872, 1080) y=(463872,) | dt=5.46s
[02:06:04]   [TRAIN] Entrenando Ridge ...
[02:06:14]   [RIDGE-FIT] extract=0.00s | cast=2.18s | fit=7.61s | total=9.78s | solver=lsqr | X=(463872, 1080) float32
[02:06:14]   [TRAIN] OK | dt=9.79s
[02:06:14]   [BUILD] Creando bundle EVAL (valid/test, flatten_X=True) ...
H60 Valid: (99328, 1080) (99328,)
H60 Test : (99840, 1080) (99840,)
Scaler H60: StandardScaler
[02:06:16]   [BUILD] OK | valid X=(99328, 1080) y=(99328,) | test X=(99840, 1080) y=(99840,) | dt=1.64s
[02:06:16]   [

/tmp/ipykernel_4084/3430405255.py:100: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_new], ignore_index=True)


[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/seq2one_ridge_metrics.parquet
[02:07:19]   [SAVE] checkpoint OK | dt=0.14s
[02:07:19] [1/5] [DONE] L30 | dt_total=80.50s | filter=0.00s | run=80.35s | dedupe=0.01s | merge=0.00s | save=0.14s

[02:07:19] [2/5] [RUN] RIDGE incremental | L60 | alpha=1.0

RIDGE | SEQ2ONE | WINDOW_SIZE=L60 | alpha=1.0
[02:07:19] [1/4] START target='delta_60' | L60
[02:07:19]   [BUILD] Creando bundle TRAIN (flatten_X=True) ...
H60 Train: (436692, 2160) (436692,)
Scaler H60: StandardScaler
[02:07:26]   [BUILD] OK | train X=(436692, 2160) y=(436692,) | dt=6.22s
[02:07:26]   [TRAIN] Entrenando Ridge ...
[02:07:48]   [RIDGE-FIT] extract=0.00s | cast=3.68s | fit=18.51s | total=22.19s | solver=lsqr | X=(436692, 2160) float32
[02:07:48]   [TRAIN] OK | dt=22.20s
[02:07:48]   [BUILD] Creando bundle EVAL (valid/test, flatten_X=True) ...
H60 Valid: (93508, 2160) (93508,)
H60 Test : (93990, 2160) (93990,)
Scaler H60: StandardScaler


,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,alpha
0,ridge,test,30,delta_60,60,54.010537,83.702586,-0.009366,0.475760,1.0
1,ridge,valid,30,delta_60,60,35.659430,50.395482,-0.007433,0.466609,1.0
2,ridge,test,30,delta_90,90,67.989720,104.424386,-0.008398,0.473722,1.0
3,ridge,valid,30,delta_90,90,44.797038,63.141213,-0.009706,0.468550,1.0
4,ridge,test,30,ret_60,60,0.002715,0.004376,-0.044519,0.474445,1.0
5,ridge,valid,30,ret_60,60,0.002007,0.002801,-0.018225,0.463500,1.0
6,ridge,test,30,ret_90,90,0.003427,0.005444,-0.034613,0.474174,1.0
7,ridge,valid,30,ret_90,90,0.002533,0.003517,-0.024712,0.465088,1.0
8,ridge,test,60,delta_60,60,55.561429,85.678667,-0.009296,0.475518,1.0
9,ridge,valid,60,delta_60,60,36.871409,51.648570,-0.007800,0.472353,1.0


In [30]:
display(df_ridge_all_sizes)

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,alpha
0,ridge,test,30,delta_60,60,54.010537,83.702586,-0.009366,0.475760,1.0
1,ridge,valid,30,delta_60,60,35.659430,50.395482,-0.007433,0.466609,1.0
2,ridge,test,30,delta_90,90,67.989720,104.424386,-0.008398,0.473722,1.0
3,ridge,valid,30,delta_90,90,44.797038,63.141213,-0.009706,0.468550,1.0
4,ridge,test,30,ret_60,60,0.002715,0.004376,-0.044519,0.474445,1.0
5,ridge,valid,30,ret_60,60,0.002007,0.002801,-0.018225,0.463500,1.0
6,ridge,test,30,ret_90,90,0.003427,0.005444,-0.034613,0.474174,1.0
7,ridge,valid,30,ret_90,90,0.002533,0.003517,-0.024712,0.465088,1.0
8,ridge,test,60,delta_60,60,55.561429,85.678667,-0.009296,0.475518,1.0
9,ridge,valid,60,delta_60,60,36.871409,51.648570,-0.007800,0.472353,1.0
